# CNN Training + Model Export

Clean 4-class training notebook using 224×224 images. Each section performs one step.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model

## 2. Configuration

In [ ]:
DATA_DIR = Path("path/to/input/training_images")
MODEL_PATH = Path("path/to/model/actualcnn4_psh_9.h5")

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
SEED = 123
EPOCHS = 8
NUM_CLASSES = 4

CLASS_NAMES = [
    "No_Growth",
    "No_PSH",
    "PSH",
    "Contamination",
]

## 3. Preprocess images

In [ ]:
def preprocess_images(images, labels, training=False):
    images = tf.cast(images, tf.float32)
    images = tf.image.adjust_contrast(images, 2.0)

    if training:
        images = tf.image.random_brightness(images, max_delta=0.1)

    images = images / 255.0
    return images, labels

## 4. Load training and validation data

In [ ]:
def load_dataset(subset, training=False):
    dataset = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        validation_split=0.2,
        subset=subset,
        seed=SEED,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_names=CLASS_NAMES,
    )

    class_names = dataset.class_names
    dataset = dataset.map(
        lambda images, labels: preprocess_images(images, labels, training),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset, class_names


train_ds, class_names = load_dataset("training", training=True)
val_ds, val_class_names = load_dataset("validation")

assert class_names == val_class_names == CLASS_NAMES
print("Class names:", class_names)

## 5. Build model

In [ ]:
def create_model():
    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer="adam",
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model


model = create_model()
model.summary()

## 6. Train model

In [ ]:
def train_model(model):
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
    )

    return model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[early_stopping],
    )


history = train_model(model)

## 7. Evaluate model

In [ ]:
val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
train_loss, train_accuracy = model.evaluate(train_ds, verbose=0)

print(f"Validation accuracy: {val_accuracy:.2%}")
print(f"Training accuracy: {train_accuracy:.2%}")

## 8. Confusion matrix

In [ ]:
def plot_confusion_matrix(model, dataset, class_names):
    true_labels = []
    predicted_labels = []

    for images, labels in dataset:
        probabilities = model.predict(images, verbose=0)
        true_labels.extend(labels.numpy())
        predicted_labels.extend(np.argmax(probabilities, axis=1))

    cm = tf.math.confusion_matrix(
        true_labels,
        predicted_labels,
        num_classes=len(class_names),
    ).numpy()

    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    fig.colorbar(image, ax=ax)

    ax.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        xlabel="Predicted label",
        ylabel="True label",
        title="Confusion Matrix",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    threshold = cm.max() / 2 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center",
                color="white" if cm[i, j] > threshold else "black",
            )

    fig.tight_layout()
    plt.show()


plot_confusion_matrix(model, val_ds, class_names)

## 9. Export model

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")